# Run Age at Onset (AAO) Regression for ITSN1 Variants vs PD Carrier AAO

## Imports

In [ ]:
import pandas as pd
import subprocess

## Prepare input files

In [ ]:
# make new covariate and samples to keep files with 0 for the FID for WGS

CHROM = "chr21"
GENE = "ITSN1"
for ANCESTRY in ["AAC", "AFR", "AJ", "AMR", "CAS", "EAS", "EUR", "MDE", "SAS"]:
    PREFIX = f"{ANCESTRY}"
    
    WORK_DIR =  f'/home/jupyter/ITSN1_R11_Results/{GENE}_WGS'
    OUTPUT_DIR = f'/home/jupyter/ITSN1_R11_Results/PLINK_GLM_files/WGS'
    ! mkdir -p {OUTPUT_DIR}
    covariate = f'{WORK_DIR}/{ANCESTRY}_covariate_file.txt'
    covar_df = pd.read_csv(covariate, sep="\t")
    covar_df["#FID"] = 0 
    covar_df.to_csv(f"{OUTPUT_DIR}/{ANCESTRY}_covariate_noFID.txt", sep="\t", na_rep="NA", index=False)


## Run AAO Regression for Imputed and WGS data

In [ ]:

for DATA in ["IMP", "WGS"]:
    for ANCESTRY in ["AAC", "AFR", "AJ", "AMR", "CAS", "EAS", "EUR", "MDE", "FIN", "SAS"]:
        try:
            CHROM = 'chr21'
            ancestry = ANCESTRY
            GENE = 'ITSN1'
            PREFIX = f"{ANCESTRY}"
            WORK_DIR = f'/home/jupyter/ITSN1_R11_Results/{GENE}_{DATA}/{ANCESTRY}/'
            OUTPUT_DIR = f'/home/jupyter/ITSN1_R11_Results/PLINK_GLM_files/'
            ! mkdir -p {OUTPUT_DIR}
            inputPfile = f"{WORK_DIR}/{CHROM}_{ANCESTRY}"
            outputGLM = f"{OUTPUT_DIR}/{PREFIX}_{DATA}.cases_only"
            samplesToKeep = f"{WORK_DIR}/{ANCESTRY}.samplestoKeep"
            covariate = f"{WORK_DIR}/{ANCESTRY}_covariate_file.txt"
            extract = f"{WORK_DIR}/{ANCESTRY}_{GENE}.all_variants_toKeep.txt"

            plinkGLM = [
                "/home/jupyter/tools/plink2",
                "--pfile", inputPfile,
                "--keep", samplesToKeep,
                "--glm", "hide-covar",
                "cols=+a1count,+a1freqcc,+a1countcc,+beta,+totallele,+totallelecc,+gcountcc",
                "--extract", extract,
                "--pheno", covariate,
                "--pheno-name", "AAO",
                "--hwe", "0.0001",
                "--mac", "1",
                "--adjust",
                "--ci", "0.95",
                "--silent",
                "--covar", covariate,
                "--covar-variance-standardize",
                "--covar-name", "SEX,PHENO,PC1,PC2,PC3,PC4,PC5",
                "--keep-if", "PHENO==2",
                "--out", outputGLM
            ]

            # Run plink2 command as bash subprocess
            subprocess.run(plinkGLM, check=True)
            
            # join glm and adjusted glm file
            glm = pd.read_csv(f"{OUTPUT_DIR}/{PREFIX}_{DATA}.cases_only.AAO.glm.linear", sep="\t")
            glm_adjust = pd.read_csv(f"{OUTPUT_DIR}/{PREFIX}_{DATA}.cases_only.AAO.glm.linear.adjusted", sep="\t", usecols=["ID", "BONF"])
            full_glm_results = pd.merge(glm, glm_adjust, on="ID", how="left")
            full_glm_results.insert(0, "Ancestry", ancestry)
            OUTPUT_DIR = f'/home/jupyter/ITSN1_R11_Results/AAO_Regression/'
            ! mkdir -p {OUTPUT_DIR}
            full_glm_results.to_csv(f"{OUTPUT_DIR}/{ancestry}_{GENE}_AAO_glm_results_{DATA}.txt", sep="\t", index=False)

        except Exception as e:
            print(f"Error processing {DATA}, {ANCESTRY}: {e}")